# cvfoodid -- Train YOLOv8 on Kaggle (free P100/T4, 30 GPU-h/week)

Companion to `01_train_colab.ipynb`. Same scripts, same dataset, different host. Use this when Colab Free GPU quota is exhausted or you want a more reliable runtime (Kaggle quota resets weekly, Saturday 00:00 UTC).

## One-time Kaggle setup

1. **Login** at https://www.kaggle.com -> verify phone in **Settings -> Account** (mandatory for GPU access).
2. **Code -> + New Notebook** (top-left).
3. **Sidebar (right) -> Notebook options:**
   - **Accelerator:** `GPU P100` (preferred, ~30 % faster than T4) or `GPU T4 x2` if P100 is busy.
   - **Internet:** **On** (default off; required for git clone, pip install, and the HuggingFace dataset download).
   - **Persistence:** `Files only` (so `/kaggle/working/` survives across sessions in this notebook).
4. **Save the notebook once** before the first long run (top-right **Save Version -> Quick Save**) so Kaggle starts tracking output.

**Each Kaggle session caps at 9 hours** (vs. Colab Free's 12). With `yolov8n --epochs 60 --imgsz 640 --batch 16` on P100 that's well under the cap (~50-70 min training + ~10 min setup).

## 1. Verify GPU + clone repo

Re-run from the top after every reconnect. The clone step is idempotent (skips if already on disk).

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
import os
import subprocess

WORKDIR = '/kaggle/working/cv-food-id2'
if not os.path.isdir(WORKDIR):
    subprocess.run(
        ['git', 'clone',
         'https://github.com/OyenCar/cv-food-id2.git', WORKDIR],
        check=True,
    )
else:
    subprocess.run(['git', '-C', WORKDIR, 'pull', '--ff-only'], check=False)
os.chdir(WORKDIR)
print('cwd:', os.getcwd())
!pip install --quiet -e ".[ml]"

## 2. Download FoodSeg103

Pulled from the HuggingFace mirror `EduardoPacheco/FoodSeg103` -- same as Colab. ~1.25 GB parquet -> ~7 k images + masks under `data/raw/foodseg103/FoodSeg103/`. Idempotent: re-run is a no-op after the first success.

**Alternative:** if you've already attached a FoodSeg103 mirror via the Kaggle sidebar (e.g. `+ Add Data -> Search 'foodseg103'`), the files appear under `/kaggle/input/<dataset-slug>/`. In that case, skip this cell and symlink the input directory before cell 3 (`os.symlink('/kaggle/input/<slug>/Images', 'data/raw/foodseg103/FoodSeg103/Images')`).

In [ ]:
!python scripts/download_datasets.py foodseg103

### Optional -- Kaggle add-on datasets (Padang, Indonesian Mendeley)

On Kaggle the `kaggle` CLI inside the notebook is already authenticated against the notebook owner's account, so no `kaggle.json` upload is needed. The cell below downloads Padang Cuisine and the Indonesian Mendeley mirror via the same `download_datasets.py` command -- only succeeds when Internet is **On** in the sidebar.

In [ ]:
import subprocess

for name in ('padang-cuisine', 'indonesian-traditional-foods'):
    rc = subprocess.run(
        ['python', 'scripts/download_datasets.py', name]
    ).returncode
    if rc != 0:
        print(f'{name}: download failed (Internet off? Kaggle ToS not accepted?). Skipping.')

## 3. Convert masks to YOLO bbox labels

Idempotent: re-run is fast if `data/processed/yolo/` is already populated.

In [ ]:
!python scripts/prepare_yolo_dataset.py \
    --raw data/raw/foodseg103/FoodSeg103 \
    --out data/processed/yolo \
    --data-yaml configs/data.yaml
!ls data/processed/yolo/images/train | head
!ls data/processed/yolo/labels/train | head

## 4. Train YOLOv8n

Defaults: **60 epochs, 640 imgsz, batch 16, `cache=disk`, `patience=15`, `save_period=10`** -- tuned for P100.

Outputs land in `/kaggle/working/runs/cvfoodid-yolo/` and persist across reconnects (no Drive mount needed). Periodic checkpoints every 10 epochs go to `weights/epoch{N}.pt` so a session timeout loses at most 10 epochs of work; resume via the next cell.

In [ ]:
import os

RUNS_DIR = '/kaggle/working/runs'
os.makedirs(RUNS_DIR, exist_ok=True)
os.environ['RUNS_DIR'] = RUNS_DIR
print('weights will land in:', RUNS_DIR)

In [ ]:
!python scripts/train_yolo.py \
    --data configs/data.yaml \
    --model yolov8n.pt \
    --epochs 60 \
    --imgsz 640 \
    --batch 16 \
    --cache disk \
    --patience 15 \
    --save-period 10 \
    --project "$RUNS_DIR" \
    --name cvfoodid-yolo

### Resume training (after a Kaggle reconnect)

Same command as above plus `--resume`. Auto-detects `$RUNS_DIR/cvfoodid-yolo/weights/last.pt`; if missing the flag is ignored and training starts fresh. Safe to leave in the notebook.

In [ ]:
!python scripts/train_yolo.py \
    --data configs/data.yaml \
    --model yolov8n.pt \
    --epochs 60 \
    --imgsz 640 \
    --batch 16 \
    --cache disk \
    --patience 15 \
    --save-period 10 \
    --project "$RUNS_DIR" \
    --name cvfoodid-yolo \
    --resume

## 5. Evaluate

In [ ]:
import os

from ultralytics import YOLO

best = f"{os.environ['RUNS_DIR']}/cvfoodid-yolo/weights/best.pt"
print('evaluating', best)
model = YOLO(best)
metrics = model.val(data='configs/data.yaml', imgsz=640)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

## 6. Export TFLite (INT8) for mobile

In [ ]:
import os
import pathlib
import shutil
import subprocess

best = f"{os.environ['RUNS_DIR']}/cvfoodid-yolo/weights/best.pt"
best_dir = pathlib.Path(best).parent
print(f'exporting from {best}')

!python scripts/export_tflite.py \
    --weights "$best" \
    --imgsz 640 \
    --calib data/processed/yolo/images/val \
    --int8

# Ultralytics writes .tflite next to source; if it landed in a temp
# location (e.g. runs/detect/...), copy any new .tflite next to best.pt.
for tf in pathlib.Path('runs/detect/cvfoodid-yolo/weights').glob('*.tflite'):
    dst = best_dir / tf.name
    if not dst.exists():
        shutil.copy(tf, dst)
        print(f'copied {tf} -> {dst}')

subprocess.run(['ls', '-lh', str(best_dir)])

## 7. Save the notebook output (CRITICAL)

Kaggle persists `/kaggle/working/` **only if you click Save Version**. Without that step the weights vanish when the session ends, just like Colab without Drive.

1. Top-right -> **Save Version**.
2. Pick **Quick Save** (just snapshot the current notebook + `/kaggle/working/`) or **Save & Run All** (re-runs every cell from scratch -- only do this if you have time to re-train).
3. After save completes, the notebook's **Output** tab will list `runs/cvfoodid-yolo/weights/best.pt` (and `best_int8.tflite` if you ran cell 6). Click any file to download.

**Pro tip:** to download just the TFLite without re-running, use the Files panel on the right -> right-click `best_int8.tflite` -> Download.

## 8. Inference + nutrition computation

Continue in `notebooks/02_inference_demo.ipynb` -- works locally with the downloaded `best.pt`.